### Importación de librerías


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import graphviz
from tabulate import tabulate
from itertools import product

# Modelos de Machine Learning
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import xgboost as xgb

# Preprocesamiento
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Model Selection y Validación
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve, validation_curve
)

# Métricas
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
)

### Cargar datos

In [6]:
df_limpio = pd.read_csv('../data/stg/IA_PROPENSITY_TRAIN_1.csv', index_col=0) 
df_limpio.head()

,TIPO_CARROCERIA,COMBUSTIBLE,Potencia,TRANS,FORMA_PAGO,ESTADO_CIVIL,GENERO,OcupaciOn,PROVINCIA,Campanna1,...,Zona_Renta,REV_Garantia,Averia_grave,QUEJA_CAC,COSTE_VENTA,km_anno,Mas_1_coche,Revisiones,Edad_Cliente,Tiempo
PRODUCTO,,,,,,,,,,,,,,,,,,,,,
0,0,0,1,1,0,0,1,1,4,1,...,2,0,2,1,2892,0,0,2,18,0
0,0,0,1,1,0,0,0,1,47,0,...,2,1,3,0,1376,7187,0,2,53,0
0,0,0,1,1,3,0,1,1,30,0,...,1,0,3,0,1376,0,1,4,21,3
0,0,0,1,1,2,0,0,1,32,1,...,1,1,2,1,2015,7256,1,4,48,5
0,0,0,1,1,2,0,0,2,41,1,...,0,0,3,0,1818,0,1,3,21,3


In [7]:
X = df_limpio.drop(columns=['Tiempo', 'Mas_1_coche']) 
y = df_limpio['Mas_1_coche']  # Objetivo

# Dividir los datos en entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Random Forest

In [ ]:
# Definir los parámetros de los modelos

param_grid = {
    'criterion': ['entropy'],
    'n_estimators': [500],
    'max_depths' : [3, 10],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5]
}

# Almacenar los resultados

results = []

# Iterar sobre los parámetros

for params in product(*param_grid.values()):
    criterion, n_estimators, max_depth, min_samples_split, min_samples_leaf = params
    model = RandomForestClassifier(
        criterion=criterion,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight='balanced',
        random_state=42
    )

    # Entrenar el modelo

    model.fit(X_train, y_train)

    # Predecir el modelo en los datos de prueba

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > 0.4).astype(int)

    # Calcular las métricas

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Validación cruzada

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')
    mean_cv_score = np.mean(cv_scores)
    
    # Guardar los resultados

    results.append({
        'criterion': criterion,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'n_estimators': n_estimators,
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,  # <-- Priorizar recall
        'roc_auc': roc_auc,
        'cv_recall': mean_cv_score  # <-- Guardar recall en validación cruzada
    })


# Convertir los resultados en un DataFrame

results_df = pd.DataFrame(results).sort_values(by=['recall', 'f1_score'], ascending=False)

# Mostrar los resultados

display(results_df.head(5))


### XGBoost